# Knapsack solved with 5 techniques

### Problem definition
Simple Knapsack problem
- Maximize total score (each item has a score)
- Volume and weight as constraints

### Solutions
1. Solve it using an Exact optimization technique
2. Genetic algorithm
3. Simulated Annealing
4. Brute force solution
5. Constructive heuristic

#### Results and Summary at the end

In [2]:
import gurobipy as gp
from gurobipy import GRB
import random
import math
import matplotlib.pyplot as plt
import itertools
import numpy as np

ModuleNotFoundError: No module named 'matplotlib'

## Create simple Knapsack problem

hardcoded knapsack

first 15 items

In [ ]:
# list of items, with weight, volume, and value
items = [
    {"w": 1, "vol": 3, "val": 2},
    {"w": 4, "vol": 4, "val": 5},
    {"w": 3, "vol": 8, "val": 7},
    {"w": 6, "vol": 7, "val": 8},
    {"w": 3, "vol": 5, "val": 3},
    {"w": 2, "vol": 4, "val": 2},
    {"w": 6, "vol": 5, "val": 9},
    {"w": 7, "vol": 8, "val": 9},
    {"w": 4, "vol": 2, "val": 6},
    {"w": 1, "vol": 4, "val": 2},
    {"w": 6, "vol": 2, "val": 4},
    {"w": 5, "vol": 2, "val": 4},
    {"w": 3, "vol": 1, "val": 3},
    {"w": 5, "vol": 7, "val": 7},
    {"w": 3, "vol": 6, "val": 6}
]
print("Total items: ", len(items), items)

Total items:  15 [{'w': 1, 'vol': 3, 'val': 2}, {'w': 4, 'vol': 4, 'val': 5}, {'w': 3, 'vol': 8, 'val': 7}, {'w': 6, 'vol': 7, 'val': 8}, {'w': 3, 'vol': 5, 'val': 3}, {'w': 2, 'vol': 4, 'val': 2}, {'w': 6, 'vol': 5, 'val': 9}, {'w': 7, 'vol': 8, 'val': 9}, {'w': 4, 'vol': 2, 'val': 6}, {'w': 1, 'vol': 4, 'val': 2}, {'w': 6, 'vol': 2, 'val': 4}, {'w': 5, 'vol': 2, 'val': 4}, {'w': 3, 'vol': 1, 'val': 3}, {'w': 5, 'vol': 7, 'val': 7}, {'w': 3, 'vol': 6, 'val': 6}]


### Hardcoded limits and N for all solutions

In [ ]:
# Define limits and n
W_max = 30
V_max = 30
n = len(items)

## 1. Exact optimization technique

using gurobi


In [ ]:
# Create model
model = gp.Model()

# Binary variables
x = [model.addVar(vtype=GRB.BINARY, name=f"x{i}") for i in range(n)]

# Constraints
model.addConstr(gp.quicksum(items[i]['w'] * x[i] for i in range(n)) <= W_max)
model.addConstr(gp.quicksum(items[i]['vol'] * x[i] for i in range(n)) <= V_max)

# Objective: maximize total value
model.setObjective(gp.quicksum(items[i]['val'] * x[i] for i in range(n)), GRB.MAXIMIZE)

# Optimize
model.optimize()

# Get selected items
selected = [i for i in range(n) if x[i].X > 0.5]
print("Selected items:", selected)
print("Total value:", sum(items[i]['val'] for i in selected))
print("Total weight:", sum(items[i]['w'] for i in selected))
print("Total volume:", sum(items[i]['vol'] for i in selected))

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 2 rows, 15 columns and 30 nonzeros (Max)
Model fingerprint: 0x7be53706
Model has 15 linear objective coefficients
Variable types: 0 continuous, 15 integer (15 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [2e+00, 9e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+01, 3e+01]
Found heuristic solution: objective 34.0000000
Presolve time: 0.00s
Presolved: 2 rows, 15 columns, 30 nonzeros
Variable types: 0 continuous, 15 integer (15 binary)

Root relaxation: objective 4.250000e+01, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 

### Optimality gap: 0% → Gurobi confirmed this is the exact optimal solution.

**Gurobi Result:**

    Optimal solution found (tolerance 1.00e-04)
    Best objective 4.100000000000e+01, best bound 4.100000000000e+01, gap 0.0000%
    Selected items: [1, 2, 6, 8, 11, 12, 13]
    Total value: 41
    Total weight: 30
    Total volume: 29

## 2. Genetic algorithm

In [ ]:
# function for fitness (score)
def fitness(individual, items, W_max, V_max):
    total_weight = sum(items[i]['w'] * bit for i, bit in enumerate(individual))
    total_volume = sum(items[i]['vol'] * bit for i, bit in enumerate(individual))
    total_value = sum(items[i]['val'] * bit for i, bit in enumerate(individual))

    # Penalize if constraints are violated
    if total_weight > W_max or total_volume > V_max:
        return 0  # or negative penalty
    return total_value

In [ ]:
# create solutions (population)
def initialize_population(pop_size, n):
    return np.random.randint(0, 2, size=(pop_size, n))

In [ ]:
# selection (pick the best)
def tournament_selection(pop, fitnesses, k=3):
    idx = np.random.choice(len(pop), k)
    best_idx = idx[np.argmax(fitnesses[idx])]
    return pop[best_idx].copy()

In [ ]:
# create child
def crossover(parent1, parent2):
    point1, point2 = sorted(np.random.choice(len(parent1), 2, replace=False))
    child1 = parent1.copy()
    child2 = parent2.copy()
    child1[point1:point2] = parent2[point1:point2]
    child2[point1:point2] = parent1[point1:point2]
    return child1, child2

In [ ]:
# mutation (random changes for diversity)
def mutate(ind, mutation_prob=0.05):
    for i in range(len(ind)):
        if np.random.rand() < mutation_prob:
            ind[i] = 1 - ind[i]  # flip bit

### Full **Genetic algorithm run**

In [ ]:
# Parameters
pop_size = 40
generations = 200
mutation_prob = 0.05
n = len(items)

# create inital population
population = initialize_population(pop_size, n)

for gen in range(generations):
    fitnesses = np.array([fitness(ind, items, W_max, V_max) for ind in population])
    new_pop = []
    while len(new_pop) < pop_size:
        # Selection
        parent1 = tournament_selection(population, fitnesses)
        parent2 = tournament_selection(population, fitnesses)
        # Crossover
        child1, child2 = crossover(parent1, parent2)
        # Mutation
        mutate(child1, mutation_prob)
        mutate(child2, mutation_prob)
        new_pop.extend([child1, child2])
    population = np.array(new_pop[:pop_size])

# Best solution
fitnesses = np.array([fitness(ind, items, W_max, V_max) for ind in population])
best_idx = np.argmax(fitnesses)
best_ind = population[best_idx]

print("Selected items:", np.where(best_ind==1)[0])
print("Total value:", fitnesses[best_idx])

Selected items: [ 3  6  7  8 12 14]
Total value: 41


## 3. Simulated Anneling

Simulated Anneling

initial temperature = 100,

minimum temperature = 0.01,

alpha = 0.99

In [ ]:
def solve_knapsack_sa(items, n, max_w, max_v, t_init=100.0, t_min=0.01, alpha=0.99):
    """
    Simulated anneling optimization algorithm.

    Parameters:
      weights = list of knapsack items weights limits.
      volumes = list of knapsack items volume limits.
      values = list of knapsack items values.
      n = number of items.
      max_w = weight capacity.
      max_v = volume capacity.
      t_init = initial temperature.
      t_min = when to stop polishing temperature.
      alpha = temperature decay.
    """

    # Start with all items as 0, empty bag
    current_state = [0] * n

    # Nested function for calculating score
    def get_totals(state):
        val_total = sum(items[i]['val'] for i, bit in enumerate(state) if bit) # if bit == 1, include that item in the Sum
        w_total = sum(items[i]['w'] for i, bit in enumerate(state) if bit)
        vol_total = sum(items[i]['vol'] for i, bit in enumerate(state) if bit)
        return val_total, w_total, vol_total

    # Initialize for anneling loop: current value, best states, best value, and initial temperature
    current_val, _, _ = get_totals(current_state)
    best_state = list(current_state)
    best_val = current_val
    T = t_init

    # Main anneling loop
    while T > t_min:
        for _ in range(50):  # Inner loops per temperature
            # Create neighbor by flipping one random item
            idx = random.randint(0, n - 1)
            next_state = list(current_state)
            next_state[idx] = 1 - next_state[idx]

            # Repair: If over capacity, remove random items until it fits
            val, w, vol = get_totals(next_state)
            while w > max_w or vol > max_v:
                # Find an item currently in the bag and remove it
                active_items = [i for i, bit in enumerate(next_state) if bit == 1]
                if not active_items: break
                remove_idx = random.choice(active_items)
                next_state[remove_idx] = 0
                val, w, vol = get_totals(next_state)

            # Acceptance Logic
            delta = val - current_val
            # Accept if better, or accept if worse based on Temperature
            if delta > 0 or random.random() < math.exp(delta / T):
                current_state = next_state
                current_val = val

                if current_val > best_val:
                    best_val = current_val
                    best_state = list(current_state)

        T *= alpha  # Cool down

    return best_val, best_state

In [ ]:
# use SA function on our hardcoded knapsack problem
best_val, best_state = solve_knapsack_sa(
    items,
    n,
    W_max,
    V_max,
    100,
    0.01,
    0.99
)
print("best value and state:", best_val, best_state)

best value and state: 41 [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1]


## 4. Brute Force

1. generate solution
2. calculate score
3. check if feasible
4. compare scores
5. repeat until every possibility is checked.

In [ ]:
def solve_knapsack_brute_force(items, n, max_w, max_v):
    """
    Brute force optimization algorithm.
    Parameters:
      weights = list of knapsack items weights limits.
      volumes = list of knapsack items volume limits.
      values = list of knapsack items values.
      n = number of items.
      max_w = weight capacity.
      max_v = volume capacity.
    """
    best_val = 0
    best_comb = None

    # Generate all possible combinations of 0s and 1s of length n
    for comb in itertools.product([0, 1], repeat=n):
        current_w = sum(items[i]['w'] for i in range(n) if comb[i])
        current_v = sum(items[i]['vol'] for i in range(n) if comb[i])
        current_val = sum(items[i]['val'] for i in range(n) if comb[i])

        # Check if valid and better than current best
        if current_w <= max_w and current_v <= max_v:
            if current_val > best_val:
                best_val = current_val
                best_comb = comb

    return best_val, best_comb

In [ ]:
# use Brute Force on hardcoded knapsack problem
best_val, best_comb = solve_knapsack_brute_force(items, n, W_max, V_max)
print("best value and state:", best_val, best_comb)

best value and state: 41 (0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1)


## 5. Constructive heuristic

Simple single-pass

Greedy heuristic

In [ ]:
def greedy_constructive_heuristic(items, max_w, max_v):
    # 1. Score and Sort items (Highest density first)
    # We use (w + vol) as the cost
    sorted_items = sorted(
        enumerate(items), # x[1] for actual values, weights, volumes
        key=lambda x: x[1]['val'] / (x[1]['w'] + x[1]['vol']), # value / cost
        reverse=True
    )

    state = [0] * len(items)
    current_w = 0
    current_v = 0
    total_val = 0

    # 2. Single Pass: Pick what fits
    for original_idx, item in sorted_items:
        if current_w + item['w'] <= max_w and current_v + item['vol'] <= max_v:
            state[original_idx] = 1
            current_w += item['w']
            current_v += item['vol']
            total_val += item['val']

    return total_val, state

In [ ]:
# greedy heuristic on hardcoded knapsack
best_val, best_state = greedy_constructive_heuristic(items, W_max, V_max)
print("best value and state:", best_val, best_state)

best value and state: 40 [0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1]


# Results & Summary

Results and summaries written after each algorithm being created.


1. **Exact optimization**

**Result:** works well, solution created with Gurobi.

**Summary:** A simple Knapsack problem like this, was no problem for Gurobi. Not much code needed. Fast.

2. **Genetic algorithm**

**Result:** does find the optimal solution but needs ~200 generations with population size 40. Still quite fast with easier problems.

**Summary:** A lot of code needed for Genetic Algorithm, can get messy without modular functions. Not the fastest solver, but for a small problem like this it still manages to get optimal or near-optimal score.

3. **Simulated Anneling**

**Result:** Simulated anneling does work, and it did even find the optimal solution for easier problems. Quite fast, but some code required.

**Summary:** Simulated anneling code became quite large and not the simplest. It needs a function for calculating score, which is nested, but it could possible be modular.

4. **Brute Force**

**Result:** Brute force technique finds the optimal solution, easy for 10 item problem.

**Summary:** Brute force works well, but as the items grow, it gets exponentially slower. Very easy code.

5. **simple single-pass constructive heuristic**

**Result:** With just 15 items Greedy heuristic did not get the optimal solution. But it was still easy to implement, fast and very close to optimal (~1-2 off).

**Summary:** Our greedy heuristic approach, item value divided by sum of item weight and volume. Sort, and then just pick the items with best score. Simple and good way to get baseline, especially when the problem isn't very hard like this 10-15 item knapsack.


### Summary of all techniques:
I tried all techniques with different 10 and 15 item knapsack problems. I was quite surprised how well each performed, maybe we could increase items to 100, to see some real delay in results.

For how simple a single pass greedy heuristic approach is, it is really good. Comnpared to the other techniques, it really doesn't care how much the problem scales. There can be 1000 items, but this should still be fast.

## More

    Assisted by Google Gemini and ChatGPT
